In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

# Create output directory if it doesn't exist
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)

# ── Style Setup ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'axes.titlecolor':  '#e6edf3',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'font.family':      'DejaVu Sans',
    'axes.titlepad':    12,
})

COLORS = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff','#56d364','#ff7b72']

# ── Load Data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('/content/API_SP.POP.TOTL_DS2_en_csv_v2_38144.csv', skiprows=4)

# Aggregate/regional codes to exclude
AGG_CODES = {
    'WLD','ARB','EAP','EAR','EAS','ECA','ECE','ECS','EMU','EUU','FCS',
    'HIC','HPC','IBD','IBT','IDA','IDB','IDX','LAC','LCN','LDC','LIC',
    'LMC','LMY','LTE','MEA','MIC','MNA','NAC','OED','OSS','PRE','PSS',
    'PST','SAS','SSA','SSF','SST','TEA','TEC','TLA','TMN','TSA','TSS',
    'UMC','AFE','AFW','CEB','CSS'
}
countries = df[~df['Country Code'].isin(AGG_CODES)].copy()

# ── FIGURE 1: Top 10 Most Populous Countries 2023 (Bar Chart) ─────────────────
fig1, ax1 = plt.subplots(figsize=(12, 6))
fig1.patch.set_facecolor('#0d1117')

top10 = (countries[['Country Name','2023']]
         .dropna()
         .nlargest(10, '2023')
         .reset_index(drop=True))
top10['pop_B'] = top10['2023'] / 1e9

bars = ax1.bar(top10['Country Name'], top10['pop_B'],
               color=COLORS[:10], edgecolor='#0d1117', linewidth=0.5,
               width=0.65, zorder=3)

# Value labels on bars
for bar, val in zip(bars, top10['pop_B']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}B', ha='center', va='bottom',
             fontsize=9.5, color='#e6edf3', fontweight='bold')

ax1.set_title('Top 10 Most Populous Countries (2023)', fontsize=15, fontweight='bold', color='#e6edf3')
ax1.set_ylabel('Population (Billions)', fontsize=11)
ax1.set_xlabel('')
ax1.yaxis.grid(True, zorder=0)
ax1.set_axisbelow(True)
ax1.tick_params(axis='x', rotation=30)
ax1.spines[['top','right','left','bottom']].set_visible(False)

# Subtitle
ax1.text(0.5, 1.04, 'Source: World Bank · SP.POP.TOTL Dataset',
         transform=ax1.transAxes, ha='center', fontsize=9, color='#8b949e')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/chart1_top10_bar.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("Chart 1 saved ✓")

# ── FIGURE 2: Population by World Region 2023 (Horizontal Bar) ────────────────
fig2, ax2 = plt.subplots(figsize=(11, 6))
fig2.patch.set_facecolor('#0d1117')

REGION_MAP = {
    'EAS': 'East Asia & Pacific',
    'SAS': 'South Asia',
    'SSA': 'Sub-Saharan Africa',
    'ECS': 'Europe & Central Asia',
    'LCN': 'Latin America & Caribbean',
    'MEA': 'Middle East & N. Africa',
    'NAC': 'North America',
}
regions_df = df[df['Country Code'].isin(REGION_MAP.keys())].copy()
regions_df['Region'] = regions_df['Country Code'].map(REGION_MAP)
regions_df = regions_df[['Region','2023']].dropna().sort_values('2023', ascending=True)
regions_df['pop_B'] = regions_df['2023'] / 1e9

colors_region = ['#79c0ff','#58a6ff','#3fb950','#56d364','#ffa657','#f78166','#d2a8ff']

hbars = ax2.barh(regions_df['Region'], regions_df['pop_B'],
                 color=colors_region, edgecolor='#0d1117',
                 height=0.6, zorder=3)

for bar, val in zip(hbars, regions_df['pop_B']):
    ax2.text(val + 0.03, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}B', va='center', fontsize=10,
             color='#e6edf3', fontweight='bold')

ax2.set_title('World Population by Region (2023)', fontsize=15, fontweight='bold', color='#e6edf3')
ax2.set_xlabel('Population (Billions)')
ax2.xaxis.grid(True, zorder=0)
ax2.set_axisbelow(True)
ax2.spines[['top','right','left','bottom']].set_visible(False)
ax2.set_xlim(0, regions_df['pop_B'].max() * 1.18)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/chart2_regions_bar.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("Chart 2 saved ✓")

# ── FIGURE 3: Histogram — Distribution of Country Populations 2023 ────────────
fig3, ax3 = plt.subplots(figsize=(12, 6))
fig3.patch.set_facecolor('#0d1117')

pop_data = countries['2023'].dropna()
pop_log = np.log10(pop_data)

n, bins, patches = ax3.hist(pop_log, bins=30, color='#58a6ff',
                             edgecolor='#0d1117', linewidth=0.6,
                             alpha=0.85, zorder=3)

# Color gradient on bars
cmap = plt.cm.cool
bin_norm = (bins[:-1] - bins[:-1].min()) / (bins[:-1].max() - bins[:-1].min())
for patch, norm_val in zip(patches, bin_norm):
    patch.set_facecolor(cmap(norm_val * 0.8 + 0.1))

# Mean/Median lines
mean_log = pop_log.mean()
median_log = pop_log.median()
ax3.axvline(mean_log, color='#f78166', linewidth=2, linestyle='--', label=f'Mean: {10**mean_log/1e6:.1f}M', zorder=4)
ax3.axvline(median_log, color='#3fb950', linewidth=2, linestyle='--', label=f'Median: {10**median_log/1e6:.1f}M', zorder=4)

# X-axis: show real population labels
xtick_vals = [4, 5, 6, 7, 8, 9, 10]
xtick_labels = ['10K','100K','1M','10M','100M','1B','10B']
ax3.set_xticks(xtick_vals)
ax3.set_xticklabels(xtick_labels)

ax3.set_title('Distribution of Countries by Population Size (2023)', fontsize=15, fontweight='bold', color='#e6edf3')
ax3.set_xlabel('Country Population (log scale)')
ax3.set_ylabel('Number of Countries')
ax3.yaxis.grid(True, zorder=0)
ax3.set_axisbelow(True)
ax3.spines[['top','right','left','bottom']].set_visible(False)
ax3.legend(fontsize=10, facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/chart3_histogram.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("Chart 3 saved ✓")

# ── FIGURE 4: World Population Growth Line Chart ──────────────────────────────
fig4, ax4 = plt.subplots(figsize=(12, 6))
fig4.patch.set_facecolor('#0d1117')

world_row = df[df['Country Code'] == 'WLD'].iloc[0]
years = list(range(1960, 2025))
world_pop = [float(world_row.get(str(y), np.nan)) / 1e9 for y in years]

valid = [(y, p) for y, p in zip(years, world_pop) if not np.isnan(p)]
yr, pop = zip(*valid)

ax4.fill_between(yr, pop, alpha=0.15, color='#58a6ff')
ax4.plot(yr, pop, color='#58a6ff', linewidth=2.5, zorder=3)
ax4.scatter([yr[-1]], [pop[-1]], color='#f78166', s=80, zorder=5)
ax4.annotate(f'{pop[-1]:.2f}B (2024)', xy=(yr[-1], pop[-1]),
             xytext=(-60, 12), textcoords='offset points',
             color='#f78166', fontsize=10, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#f78166', lw=1.5))

# Milestone markers
milestones = [(1974, 4.0, '4B'), (1987, 5.0, '5B'), (1999, 6.0, '6B'),
              (2011, 7.0, '7B'), (2022, 8.0, '8B')]
for my, mp, label in milestones:
    ax4.axhline(mp, color='#30363d', linewidth=0.8, linestyle=':')
    ax4.text(1961, mp + 0.04, label, color='#8b949e', fontsize=9)

ax4.set_title('World Population Growth (1960–2024)', fontsize=15, fontweight='bold', color='#e6edf3')
ax4.set_ylabel('Population (Billions)')
ax4.set_xlabel('Year')
ax4.yaxis.grid(True, zorder=0)
ax4.set_axisbelow(True)
ax4.spines[['top','right','left','bottom']].set_visible(False)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/chart4_world_growth.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("Chart 4 saved ✓")

# ── FIGURE 5: Growth Rate Bar Chart (top fastest & slowest) ───────────────────
fig5, ax5 = plt.subplots(figsize=(13, 6))
fig5.patch.set_facecolor('#0d1117')

countries['pop_1960'] = pd.to_numeric(countries['1960'], errors='coerce')
countries['pop_2023'] = pd.to_numeric(countries['2023'], errors='coerce')
countries = countries.dropna(subset=['pop_1960','pop_2023'])
countries = countries[countries['pop_1960'] > 500000]  # filter tiny nations
countries['growth_pct'] = ((countries['pop_2023'] - countries['pop_1960']) / countries['pop_1960']) * 100

fastest = countries.nlargest(8, 'growth_pct')[['Country Name','growth_pct']]
slowest = countries.nsmallest(5, 'growth_pct')[['Country Name','growth_pct']]
combined = pd.concat([fastest, slowest]).reset_index(drop=True)

bar_colors = ['#3fb950' if v >= 0 else '#f78166' for v in combined['growth_pct']]
bars5 = ax5.bar(combined['Country Name'], combined['growth_pct'],
                color=bar_colors, edgecolor='#0d1117', linewidth=0.5,
                width=0.65, zorder=3)

for bar, val in zip(bars5, combined['growth_pct']):
    ypos = bar.get_height() + 15 if val >= 0 else bar.get_height() - 80
    ax5.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{val:.0f}%', ha='center', fontsize=9,
             color='#e6edf3', fontweight='bold')

ax5.axhline(0, color='#8b949e', linewidth=1)
ax5.set_title('Population Growth Rate by Country (1960–2023)', fontsize=15, fontweight='bold', color='#e6edf3')
ax5.set_ylabel('Growth Rate (%)')
ax5.yaxis.grid(True, zorder=0)
ax5.set_axisbelow(True)
ax5.tick_params(axis='x', rotation=35)
ax5.spines[['top','right','left','bottom']].set_visible(False)

green_patch = mpatches.Patch(color='#3fb950', label='High Growth')
red_patch   = mpatches.Patch(color='#f78166', label='Negative/Low Growth')
ax5.legend(handles=[green_patch, red_patch], facecolor='#161b22',
           edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=10)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/chart5_growth_rate.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("Chart 5 saved ✓")

print("\n✅ All 5 charts saved to /mnt/user-data/outputs/")

Chart 1 saved ✓
Chart 2 saved ✓
Chart 3 saved ✓
Chart 4 saved ✓
Chart 5 saved ✓

✅ All 5 charts saved to /mnt/user-data/outputs/
